In [1]:
import os
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
import mlflow

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("homework-03")

with mlflow.start_run():
    mlflow.log_param("param1", 5)
    mlflow.log_metric("metric1", 0.87)

print("Experiment creation and run logging succeeded.")

2025/06/02 20:36:54 INFO mlflow.tracking.fluent: Experiment with name 'homework-03' does not exist. Creating a new experiment.


🏃 View run enthused-hen-641 at: http://127.0.0.1:5000/#/experiments/1/runs/f129ec230ab34972bd6d02250dc0d7b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Experiment creation and run logging succeeded.


In [4]:
# Question 3: create a pipeline
df = pd.read_parquet('data/yellow_tripdata_2023-03.parquet')
df.shape

(3403766, 19)

In [9]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    df_features = df[categorical]
    y = df['duration'].values
    return df_features, y

In [10]:
# Usage:
df_features, y_train = read_dataframe('data/yellow_tripdata_2023-03.parquet')

dv = DictVectorizer()
train_dicts = df_features.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)  # Sparse matrix of shape (n_samples, n_one_hots)


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Instantiate a plain (ordinary least squares) linear regression
lr_model = LinearRegression()

# Fit to the training data (handles sparse input internally)
lr_model.fit(X_train, y_train)

# Inspect learned coefficients
# Note: lr_model.coef_ is an array of shape (p,), containing weights for each one-hot feature.
weights = lr_model.coef_
intercept = lr_model.intercept_

print(f"Fitted intercept: {intercept:.4f}")
print(f"Number of features (p): {len(weights)}")

Fitted intercept: 24.7722
Number of features (p): 518


In [13]:
with mlflow.start_run() as run:
    # 2.a. Activate autologging for scikit-learn
    mlflow.sklearn.autolog()  

    # 2.b. Instantiate and train the model
    lr = LinearRegression()
    lr.fit(X_train, y_train)
   
    model_uri = f"runs:/{run.info.run_id}/model"
    mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="model",
        registered_model_name="YellowTaxiDurationModel"
    )

2025/06/02 22:06:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'YellowTaxiDurationModel'.
2025/06/02 22:06:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: YellowTaxiDurationModel, version 1


🏃 View run wise-mouse-749 at: http://127.0.0.1:5000/#/experiments/1/runs/968330c3166a4812889a1ccdfd07022e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'YellowTaxiDurationModel'.
